# Parte 2 — Diagnóstico e limpeza

Diagnóstico da base, conversão de tipos, padronização de textos e remoção das colunas
de localidade fina.


## 2.0 Setup e reconstrução da base

Este notebook é autônomo: refaz a leitura da Parte 1 para poder rodar isolado.


In [1]:
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd


def localizar_raiz() -> Path:
    for candidato in [Path.cwd(), *Path.cwd().parents]:
        if (candidato / "dados_brutos").is_dir():
            return candidato
    raise FileNotFoundError("Abra o notebook da raiz do projeto ou de 'notebooks/'.")


RAIZ = localizar_raiz()
DIR_BRUTOS = RAIZ / "dados_brutos"
DIR_TRATADOS = RAIZ / "dados_tratados"
DIR_RESULTADOS = RAIZ / "resultados"
for pasta in (DIR_TRATADOS, DIR_RESULTADOS):
    pasta.mkdir(parents=True, exist_ok=True)

SEPARADOR = ";"
ENCODING_SAIDA = "utf-8-sig"
ANOS = (2023, 2024, 2025)
print(RAIZ)


/home/user/PRF-pandas-e-Plotly---2023-2025


In [2]:
def ler_csv_prf(caminho: Path) -> pd.DataFrame:
    """Lê o CSV tentando vários encodings; só falha se todos falharem."""
    for encoding in ("utf-8", "latin1", "utf-8-sig", "cp1252"):
        try:
            df = pd.read_csv(caminho, sep=SEPARADOR, encoding=encoding,
                             low_memory=False, dtype={"id": "string"})
        except (UnicodeDecodeError, UnicodeError):
            continue
        print(f"{caminho.name}: encoding={encoding}, {len(df):,} linhas")
        return df
    raise RuntimeError(f"Nenhum encoding funcionou para {caminho}")


def normalizar_nome(nome: str) -> str:
    nome = str(nome).strip().lower()
    nome = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode()
    return nome.replace(" ", "_").replace("-", "_").replace("/", "_").strip("_")


In [3]:
dados_por_ano = {}
for ano in ANOS:
    df = ler_csv_prf(DIR_BRUTOS / f"datatran{ano}.csv")
    df.columns = [normalizar_nome(c) for c in df.columns]
    df["arquivo_origem"] = ano
    dados_por_ano[ano] = df

acidentes_prf_historico = pd.concat(
    [dados_por_ano[ano] for ano in ANOS], ignore_index=True
)
print(acidentes_prf_historico.shape)


datatran2023.csv: encoding=latin1, 67,766 linhas


datatran2024.csv: encoding=latin1, 73,202 linhas


datatran2025.csv: encoding=latin1, 72,529 linhas
(213497, 31)


## 2.1 Diagnóstico: `info()`, nulos e duplicidades


In [4]:
acidentes_prf_historico.info(memory_usage="deep", show_counts=True)


<class 'pandas.DataFrame'>
RangeIndex: 213497 entries, 0 to 213496
Data columns (total 31 columns):
 #   Column                  Non-Null Count   Dtype 
---  ------                  --------------   ----- 
 0   id                      213497 non-null  string
 1   data_inversa            213497 non-null  str   
 2   dia_semana              213497 non-null  str   
 3   horario                 213497 non-null  str   
 4   uf                      213497 non-null  str   
 5   br                      213497 non-null  int64 
 6   km                      213497 non-null  str   
 7   municipio               213497 non-null  str   
 8   causa_acidente          213497 non-null  str   
 9   tipo_acidente           213497 non-null  str   
 10  classificacao_acidente  213494 non-null  str   
 11  fase_dia                213497 non-null  str   
 12  sentido_via             213497 non-null  str   
 13  condicao_metereologica  213497 non-null  str   
 14  tipo_pista              213497 non-null  str   

In [5]:
nulos = pd.DataFrame({
    "qtd": acidentes_prf_historico.isna().sum(),
    "pct": acidentes_prf_historico.isna().mean().mul(100).round(3),
}).query("qtd > 0").sort_values("qtd", ascending=False)
display(nulos)


,qtd,pct
uop,155,0.073
delegacia,91,0.043
regional,19,0.009
classificacao_acidente,3,0.001


In [6]:
duplicadas = int(acidentes_prf_historico.duplicated().sum())
df_deduplicado = acidentes_prf_historico.drop_duplicates().copy()
print(f"Duplicidades exatas: {duplicadas} | linhas removidas: "
      f"{len(acidentes_prf_historico) - len(df_deduplicado)}")


Duplicidades exatas: 0 | linhas removidas: 0


## 2.2 Conversão de tipos

Decimal com vírgula nas numéricas e `data_inversa` em **ISO** (`yyyy-mm-dd`) —
`%d/%m/%Y` viraria tudo `NaT`.


In [7]:
NUMERICAS = ["br", "km", "pessoas", "mortos", "feridos", "feridos_leves",
             "feridos_graves", "ilesos", "ignorados", "veiculos",
             "latitude", "longitude"]

df_limpo = df_deduplicado.copy()
for coluna in NUMERICAS:
    texto = df_limpo[coluna].astype("string").str.replace(",", ".", regex=False)
    df_limpo[coluna] = pd.to_numeric(texto, errors="coerce")

df_limpo["data_inversa"] = pd.to_datetime(
    df_limpo["data_inversa"], format="%Y-%m-%d", errors="coerce"
)

# nenhum valor pode ter virado nulo na conversão
faltando = df_limpo[NUMERICAS + ["data_inversa"]].isna().sum()
assert faltando.sum() == 0, f"Conversão perdeu valores:\n{faltando[faltando > 0]}"
print("Conversão ok — nenhum valor perdido.")


Conversão ok — nenhum valor perdido.


## 2.3 Verificação de linhas inválidas ou nulas

Verificação simples: cada coluna categórica é comparada com o domínio esperado.
Linha com valor **nulo** ou **fora do domínio** → `IGNORADO`.


In [8]:
DOMINIOS = {
    "uf": {"AC","AL","AM","AP","BA","CE","DF","ES","GO","MA","MG","MS",
           "MT","PA","PB","PE","PI","PR","RJ","RN","RO","RR","RS","SC",
           "SE","SP","TO"},
    "fase_dia": {"Amanhecer", "Anoitecer", "Plena Noite", "Pleno dia"},
    "tipo_pista": {"Dupla", "Múltipla", "Simples"},
    "uso_solo": {"Não", "Sim"},
    "sentido_via": {"Crescente", "Decrescente", "Não Informado"},
    "classificacao_acidente": {"Com Vítimas Fatais", "Com Vítimas Feridas", "Sem Vítimas"},
    "dia_semana": {"domingo", "segunda-feira", "terça-feira", "quarta-feira",
                   "quinta-feira", "sexta-feira", "sábado"},
}


In [9]:
# limpeza de espaços e invisíveis em todas as colunas de texto
ajustadas = 0
for coluna in df_limpo.columns:
    if str(df_limpo[coluna].dtype) in ("object", "string", "str"):
        antes = df_limpo[coluna]
        depois = (antes.astype("string")
                  .str.replace(r"[\u00a0\u200b]", " ", regex=True)
                  .str.strip()
                  .str.replace(r"\s{2,}", " ", regex=True))
        ajustadas += int((antes.notna() & (antes != depois)).sum())
        df_limpo[coluna] = depois

# o enunciado pede padronizar textos com strip + upper. Os identificadores
# (uf, br, municipio) já vêm em caixa alta da fonte, então o upper aqui é
# explícito e inócuo; a guarda de dtype evita tocar em `br`, que é numérico.
# As colunas descritivas (tipo_acidente, causa_acidente, fase_dia...) mantêm a
# caixa publicada pela PRF — é a forma usada nos rótulos de todo o relatório.
for coluna in ("uf", "br", "municipio"):
    if coluna in df_limpo.columns and str(df_limpo[coluna].dtype) in ("object", "string", "str"):
        df_limpo[coluna] = df_limpo[coluna].astype("string").str.upper()

print(f"Espaços padronizados em {ajustadas} linhas.")


Espaços padronizados em 32 linhas.


In [10]:
# nulo ou fora do domínio -> IGNORADO
correcoes = []
for coluna, dominio in DOMINIOS.items():
    invalidas = df_limpo[coluna].isna() | ~df_limpo[coluna].isin(dominio)
    n_nulos = int(df_limpo[coluna].isna().sum())
    n_invalidos = int(invalidas.sum()) - n_nulos
    df_limpo.loc[invalidas, coluna] = "IGNORADO"
    correcoes.append({"coluna": coluna, "nulos": n_nulos, "invalidos": n_invalidos})

log = pd.DataFrame(correcoes).query("nulos > 0 or invalidos > 0")
display(log if len(log) else "Nenhuma linha nula ou inválida.")


,coluna,nulos,invalidos
5,classificacao_acidente,3,0


In [11]:
# conferência: nada pode ter sobrado nulo ou inválido
# 'IGNORADO' passa a ser um valor aceito em qualquer categórica
restantes = {
    c: int((df_limpo[c].isna() | ~df_limpo[c].isin(d | {"IGNORADO"})).sum())
    for c, d in DOMINIOS.items()
}
assert sum(restantes.values()) == 0, f"Sobrou valor inválido: {restantes}"
print("OK — nenhuma linha nula ou inválida nas categóricas.")


OK — nenhuma linha nula ou inválida nas categóricas.


In [12]:
# contagens sem nulo
CONTAGEM = ["pessoas", "mortos", "feridos", "feridos_leves",
            "feridos_graves", "ilesos", "ignorados", "veiculos"]
for c in CONTAGEM:
    df_limpo[c] = df_limpo[c].fillna(0).astype("int64")
assert df_limpo[CONTAGEM].isna().sum().sum() == 0
print("Contagens sem nulo.")


Contagens sem nulo.


## 2.4 Remoção das colunas de localidade fina

Saem `latitude`, `longitude`, `regional`, `delegacia`, `uop`. Ficam `uf`, `br`, `municipio`.


In [13]:
REMOVER = ["latitude", "longitude", "regional", "delegacia", "uop"]
linhas_antes = len(df_limpo)
df_limpo = df_limpo.drop(columns=[c for c in REMOVER if c in df_limpo.columns])

assert len(df_limpo) == linhas_antes, "A remoção de colunas alterou o número de linhas"
assert {"uf", "br", "municipio"} <= set(df_limpo.columns)
print(f"Colunas: {df_limpo.shape[1]} | Linhas: {len(df_limpo):,}")
print(sorted(df_limpo.columns))


Colunas: 26 | Linhas: 213,497
['arquivo_origem', 'br', 'causa_acidente', 'classificacao_acidente', 'condicao_metereologica', 'data_inversa', 'dia_semana', 'fase_dia', 'feridos', 'feridos_graves', 'feridos_leves', 'horario', 'id', 'ignorados', 'ilesos', 'km', 'mortos', 'municipio', 'pessoas', 'sentido_via', 'tipo_acidente', 'tipo_pista', 'tracado_via', 'uf', 'uso_solo', 'veiculos']


## 2.5 Resumo


In [14]:
print(f"Linhas preservadas : {len(df_limpo):,}")
print(f"Colunas            : {df_limpo.shape[1]}")
print(f"Acidentes fatais   : {int((df_limpo['mortos'] >= 1).sum()):,}")
print(f"Total de mortos    : {int(df_limpo['mortos'].sum()):,}")


Linhas preservadas : 213,497
Colunas            : 26
Acidentes fatais   : 15,293
Total de mortos    : 17,833
